# Bài tập - Lọc ứng viên biển số xe theo tỉ lệ khung hình
**BVU - Cao học - Xử lý ảnh - Trương Đình Phúc**

Sau bước phát hiện các vùng ứng viên (candidate regions) trên ảnh (ví dụ bằng contour, MSER, ...), ta cần lọc bớt các vùng không phải biển số dựa trên tỉ lệ khung hình (aspect ratio = rộng / cao). Biển số xe thường có tỉ lệ rộng/cao nằm trong một khoảng nhất định, các vùng như cụm đèn, logo, gương chiếu hậu... thường có tỉ lệ khác biệt rõ rệt nên có thể loại bỏ.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Hàm lọc ứng viên theo tỉ lệ khung hình

In [ ]:
def filter_by_aspect_ratio(candidates, min_ratio=1.0, max_ratio=2.0):
    """
    Loc cac vung ung vien bien so dua tren ti le rong / cao.

    candidates: danh sach dict, moi phan tu co dang
        {'name': ten vung, 'width': chieu rong, 'height': chieu cao}
    min_ratio, max_ratio: khoang ti le chap nhan duoc

    Returns: (kept, rejected) - 2 danh sach ung vien duoc giu va bi loai
    """
    kept, rejected = [], []

    for c in candidates:
        w, h = c["width"], c["height"]

        if h <= 0:
            print(f"{c['name']}: chieu cao khong hop le")
            c["result"] = "LOAI BO"
            rejected.append(c)
            continue

        ratio = w / h
        c["aspect_ratio"] = ratio

        if min_ratio <= ratio <= max_ratio:
            c["result"] = "GIU LAI"
            kept.append(c)
        else:
            c["result"] = "LOAI BO"
            rejected.append(c)

    return kept, rejected

## Áp dụng trên tập ứng viên mẫu

Biển số xe máy/ô tô Việt Nam (1 dòng) thường có tỉ lệ rộng/cao khoảng **2 - 5**; biển số 2 dòng (xe máy) có tỉ lệ gần **1.2 - 2**. Ở đây minh hoạ với khoảng chấp nhận `[1.2, 2.0]` (biển số 2 dòng) — các vùng gần vuông (gương, logo) hay quá dẹt (cụm đèn) sẽ bị loại. Có thể chỉnh `min_ratio`, `max_ratio` tuỳ loại biển cần lọc.

In [ ]:
candidates = [
    {"name": "Vung A - Cum den",        "width": 150, "height": 50},
    {"name": "Vung B - Bien so",        "width": 140, "height": 100},
    {"name": "Vung C - Guong chieu hau", "width": 36,  "height": 34},
    {"name": "Vung D - Logo xe",         "width": 28,  "height": 26},
    {"name": "Vung E - Bien so mo",      "width": 180, "height": 100},
]

kept, rejected = filter_by_aspect_ratio(candidates, min_ratio=1.2, max_ratio=2.0)

print("=" * 60)
print("KET QUA LOC UNG VIEN THEO TI LE KHUNG HINH")
print("=" * 60)
for c in candidates:
    print(f"{c['name']:<22} | {c['width']}x{c['height']} "
          f"| ratio={c['aspect_ratio']:.2f} | {c['result']}")

print("-" * 60)
print(f"So vung giu lai: {len(kept)} / {len(candidates)}")
print(f"So vung loai bo: {len(rejected)} / {len(candidates)}")

## Minh hoạ trực quan các vùng ứng viên
Vẽ khung chữ nhật cho từng vùng ứng viên (theo width/height), tô màu xanh lá = giữ lại, đỏ = loại bỏ, để dễ hình dung kết quả lọc.

In [ ]:
canvas = np.ones((260, 700, 3), dtype=np.uint8) * 255
x_offset = 20

for c in candidates:
    w, h = c["width"], c["height"]
    color = (0, 150, 0) if c["result"] == "GIU LAI" else (0, 0, 200)
    top_left = (x_offset, 200 - h)
    bottom_right = (x_offset + w, 200)
    cv2.rectangle(canvas, top_left, bottom_right, color, 2)
    cv2.putText(canvas, f"{c['aspect_ratio']:.2f}", (x_offset, 210 + 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1)
    x_offset += w + 20

plt.figure(figsize=(12, 5))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title("Xanh la = GIU LAI (ung vien bien so) | Do = LOAI BO")
plt.axis("off")
plt.show()

## Nhận xét
- Việc lọc theo tỉ lệ khung hình là bước hậu xử lý (post-processing) đơn giản nhưng hiệu quả để loại nhanh các ứng viên rõ ràng không phải biển số (ví dụ cụm đèn quá dẹt, gương/logo gần vuông).
- Nhược điểm: ngưỡng `min_ratio`/`max_ratio` cố định có thể loại nhầm biển số bị nghiêng, bị che một phần, hoặc biển số nước ngoài có tỉ lệ khác chuẩn Việt Nam — trong thực tế thường kết hợp thêm các tiêu chí khác (diện tích, mật độ cạnh, tỉ lệ lấp đầy contour...) để lọc chính xác hơn.